# Movie Upscaler v1 — Kaggle T4 / ComfyUI

Two visible connected workflows: **SeedVR2 temporal restoration** and **Real-ESRGAN fast enlargement**.
Single-cell startup; no movie runs automatically. Uses **GPU 0** (one T4), leaving the second GPU unused.

1. Run **Movie_Upscaler_Model_Packs.ipynb** with **Accelerator: None**, Internet ON and private Kaggle API authentication. It automatically uploads a private Dataset and prints its link.
2. Import this notebook; select **T4 GPU**, Internet ON. After Dataset processing, attach it via **Add Input → Your Datasets**. Saved Notebook Outputs also remain supported.
3. Attach your source video (or copy an H3 output to accessible Kaggle storage). Set `source_path` in the leftmost ComfyUI node to its full path.
4. Run this single startup cell and open the printed **MOVIE STUDIO** link. It loads SeedVR2's graph directly.
5. Press **Run once** in ComfyUI. Keep frontend **Auto Queue OFF** and queue count **1**. The save node queues one next chunk until finished.
6. To pause, **Interrupt then Clear Queue**. Run the same graph/settings again to resume verified chunks. Set `auto_continue=false` for manual one-chunk-at-a-time processing.
7. Final movies and checkpoints are under `/kaggle/working/movie-studio/movie-jobs/`. **Save Version with output files** or preserve these files before ending Kaggle.

## Settings

- **SeedVR2:** 3B Q8 model, FP16 tiled VAE, CPU offloading, 5-frame temporal batches, overlap 2. Source chunks have 4 context frames on each side; these are cropped before saving, with no duplicated output frames. Context reduces boundary risks but does not guarantee invisible seams. Defaults to 2×, with an explicit 1920-pixel longest-edge cap. Increase the cap only after testing a short clip; VAE tiling does not bound transformer attention memory.
- **Real-ESRGAN:** select it from Workflows. Defaults to 2× (4× model then downsample). For **4×**, set source `scale=4` AND `ImageScaleBy.scale_by=1`. Mismatched dimensions fail before checkpoint commit. This model processes frames independently and can flicker.
- `chunk_frames=17` bounds the images in each ComfyUI execution; increasing it uses more RAM. Changing restoration, chunk, context, or target settings starts a distinct checkpoint job. Changing auto-continuation/export container does not discard restored chunks.
- Models must be attached by default. `REQUIRE_ATTACHED_MODELS=False` deliberately allows online model downloads. `ENABLE_SEEDVR2=False` runs only Real-ESRGAN and needs only its checkpoint. The image pack's VAE/Real-ESRGAN files can be reused, but its native INT8 SeedVR2 model is **not** the Q8 GGUF used here.

## Media and storage

This version supports **SDR constant-frame-rate, square-pixel video**, producing RGB8 checkpoints and H.264 YUV420 output. HDR, variable-rate input, and display rotation must be converted explicitly first. High-bit-depth SDR is reduced to 8-bit; this is not an HDR preservation pipeline. Input timestamps are checked before inference.

Frame count, rational frame rate, dimensions and audio-stream count are checked. Original audio streams are copied when supported. MP4 falls back to AAC if stream-copy fails; MKV keeps supported original audio and subtitles. Video is re-encoded at CRF 18. No frame interpolation or automatic face redrawing. Metadata/chapters are copied; arbitrary attachments/data tracks are not retained.

Lossless FFV1 chunk files cost disk space. Startup checks measured free space against a conservative full-job checkpoint estimate and reserve. Nothing automatically deletes source files, other jobs or completed checkpoints. Chunk decoding seeks with a short preroll and selects validated source timestamps, avoiding a full prefix decode for every chunk. Full-length films may exceed Kaggle time/disk quotas; start with a short representative clip.

Same-session interruptions resume automatically. Across sessions, restore the saved `movie-jobs` directory to the configured working output path before Run. Models are reused through the automatically created Dataset or saved Notebook Outputs. Software and tunnel setup still need Internet. Keep the startup cell running; anyone with its temporary URL can access the studio.

## Validation status

See `notebooks/movie_upscaler/README.md` for completed checks and remaining limits. Local CPU/media checks cannot establish T4 speed, memory fit, or visual restoration quality. No 4K or real-time performance promise is made.


In [ ]:
"""Pinned model metadata from upstream registry and HF LFS metadata."""
from pathlib import Path
import hashlib
import json
import os
import shutil
import urllib.request

ASSETS = [
    {'folder': 'SEEDVR2', 'name': 'seedvr2_ema_3b-Q8_0.gguf', 'bytes': 3660613984,
     'sha256': 'be0d60083a2051a265eb4b77f28edf494e6db67ffc250216f32b72292e5cbd96',
     'url': 'https://huggingface.co/AInVFX/SeedVR2_comfyUI/resolve/ac66d6d98fa49975d893b58c55bff7677191c862/seedvr2_ema_3b-Q8_0.gguf'},
    {'folder': 'SEEDVR2', 'name': 'ema_vae_fp16.safetensors', 'bytes': 501324814,
     'sha256': '20678548f420d98d26f11442d3528f8b8c94e57ee046ef93dbb7633da8612ca1',
     'aliases': ['seedvr2_ema_vae_fp16.safetensors'],
     'url': 'https://huggingface.co/numz/SeedVR2_comfyUI/resolve/09ced71023636e9bc8cdf9cdecfb2625d1e691e8/ema_vae_fp16.safetensors'},
    {'folder': 'upscale_models', 'name': 'RealESRGAN_x4plus.pth', 'bytes': 67040989,
     'sha256': '4fa0d38905f75ac06eb49a7951b426670021be3018265fd191d2125df9d682f1',
     'url': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'},
]

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8*1024**2), b''):
            h.update(block)
    return h.hexdigest()

def valid(path, asset):
    return path.is_file() and path.stat().st_size == asset['bytes'] and sha(path) == asset['sha256']

def find_asset(asset, roots):
    for root in roots:
        if not Path(root).exists():
            continue
        for name in [asset['name']]+asset.get('aliases', []):
            for p in Path(root).rglob(name):
                if valid(p, asset):
                    return p.resolve()
    return None

def fetch(asset, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        if valid(dest, asset):
            return dest
        raise RuntimeError(f'Existing file failed verification; preserved: {dest}')
    part = dest.with_suffix(dest.suffix+'.partial')
    offset = part.stat().st_size if part.exists() else 0
    if offset > asset['bytes']:
        raise RuntimeError(f'Oversized partial file preserved: {part}')
    if shutil.disk_usage(dest.parent).free < asset['bytes']-offset+512*1024**2:
        raise RuntimeError('Insufficient free disk for model and 512 MiB reserve.')
    if offset != asset['bytes']:
        req = urllib.request.Request(asset['url'], headers={'Range': f'bytes={offset}-'} if offset else {})
        with urllib.request.urlopen(req, timeout=120) as response:
            append = offset and response.status == 206 and response.headers.get('Content-Range', '').startswith(f'bytes {offset}-')
            if response.status == 206 and not append and offset:
                raise RuntimeError('Invalid resume range; partial file preserved.')
            if not append and offset and shutil.disk_usage(dest.parent).free < asset['bytes']+512*1024**2:
                raise RuntimeError('Server requires restart; insufficient free disk.')
            with open(part, 'ab' if append else 'wb') as f:
                shutil.copyfileobj(response, f, 8*1024**2)
                f.flush()
                os.fsync(f.fileno())
    if not valid(part, asset):
        raise RuntimeError(f'Checksum/size mismatch; partial file preserved: {part}')
    part.replace(dest)
    return dest

def build_pack(root, input_root='/kaggle/input'):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    marker = root/'manifest.json'
    marker.unlink(missing_ok=True)
    for asset in ASSETS:
        dest = root/asset['folder']/asset['name']
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            cached = find_asset(asset, [input_root])
            if cached:
                if shutil.disk_usage(dest.parent).free < asset['bytes']+512*1024**2:
                    raise RuntimeError('Not enough space to copy attached model into durable output.')
                shutil.copyfile(cached, dest)
        print('Verify / prepare:', asset['name'], flush=True)
        fetch(asset, dest)
    # Actual files, never symlinks into another notebook output.
    if any((root/a['folder']/a['name']).is_symlink() for a in ASSETS):
        raise RuntimeError('Pack must contain actual model files, not symlinks.')
    temp = marker.with_suffix('.partial')
    temp.write_text(json.dumps({'complete': True, 'assets': ASSETS}, indent=2))
    temp.replace(marker)
    print('Local model pack complete and verified.', flush=True)

def dataset_api():
    """Use private Kaggle credentials; never embed or print their values."""
    if not os.environ.get('KAGGLE_API_TOKEN') and not os.environ.get('KAGGLE_KEY'):
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
        except Exception:
            pass
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ModuleNotFoundError:
        import subprocess
        import sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'kaggle'], check=True)
        from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    return api

def save_private_dataset(pack_root, owner='', api=None):
    """Automatically create a private named Dataset, following Image Studio's saver.

    Flat hardlinks expose nested weights to Kaggle's uploader without copying or
    archiving gigabytes. A successful submission is recorded; processing is remote.
    Existing Datasets are never overwritten or made public.
    """
    import re
    root = Path(pack_root)
    manifest = json.loads((root/'manifest.json').read_text())
    if not manifest.get('complete') or manifest.get('assets') != ASSETS:
        raise RuntimeError('Refusing to upload an incomplete or different model pack.')
    for asset in ASSETS:
        path = root/asset['folder']/asset['name']
        if path.is_symlink() or not valid(path, asset):
            raise RuntimeError('Model failed verification before Dataset upload: '+asset['name'])
    api = api if api is not None else dataset_api()
    owner = owner or getattr(api, 'config_values', {}).get('username', '') or os.environ.get('KAGGLE_USERNAME', '')
    if not re.fullmatch(r'[A-Za-z0-9_-]+', owner):
        raise RuntimeError('Set DATASET_OWNER to your Kaggle username. Keep API credentials in Kaggle Secrets, never in the notebook.')
    fingerprint = hashlib.sha256(json.dumps(ASSETS, sort_keys=True).encode()).hexdigest()[:12]
    handle = owner+'/movie-upscaler-models-'+fingerprint
    url = 'https://www.kaggle.com/datasets/'+handle
    receipt = root/'dataset-upload.json'
    if receipt.exists() and json.loads(receipt.read_text()).get('handle') == handle:
        print('Previously submitted private Dataset:', url, flush=True)
        return url
    stage = root/'dataset-files'
    stage.mkdir(exist_ok=True)
    expected = {a['name'] for a in ASSETS} | {'manifest.json', 'dataset-metadata.json'}
    if any(p.name not in expected for p in stage.iterdir()):
        raise RuntimeError('Unexpected upload-staging file preserved; inspect dataset-files before retrying.')
    for asset in ASSETS:
        source = root/asset['folder']/asset['name']
        target = stage/asset['name']
        if target.exists():
            if target.is_symlink() or not valid(target, asset):
                raise RuntimeError('Existing upload-staging file failed verification; preserved: '+target.name)
        else:
            os.link(source, target)
    (stage/'manifest.json').write_text(json.dumps(manifest, indent=2))
    (stage/'dataset-metadata.json').write_text(json.dumps({
        'id': handle, 'title': 'Movie Upscaler Models '+fingerprint,
        'licenses': [{'name': 'other'}],
        'description': 'Private verified SeedVR2 and Real-ESRGAN model pack. Original weight licenses apply; no relicensing. See manifest.json for sources and SHA256 values.'
    }, indent=2))
    result = api.dataset_create_new(str(stage), public=False, quiet=False, convert_to_csv=False, dir_mode='skip')
    error = result.get('error') if isinstance(result, dict) else getattr(result, 'error', None)
    if result is None or error:
        raise RuntimeError('Dataset upload was not confirmed. Verified files are retained; no success receipt was written.')
    temp = receipt.with_suffix('.partial')
    temp.write_text(json.dumps({'handle': handle, 'url': url, 'submitted': True, 'requested_private': True,
                                'manifest_fingerprint': fingerprint}, indent=2))
    temp.replace(receipt)
    print('PRIVATE DATASET SUBMITTED:', url, flush=True)
    print('Wait for Kaggle processing, then attach it through Add Input → Your Datasets. No PC download or manual upload is needed.', flush=True)
    return url

def build_and_save(root, owner='', auto_create_dataset=True, input_root='/kaggle/input', api=None):
    # Fail authentication before spending time downloading the full model pack.
    if auto_create_dataset:
        import re
        if api is None:
            api = dataset_api()
        owner = owner or getattr(api, 'config_values', {}).get('username', '') or os.environ.get('KAGGLE_USERNAME', '')
        if not re.fullmatch(r'[A-Za-z0-9_-]+', owner):
            raise RuntimeError('Set DATASET_OWNER to your Kaggle username before building. No model downloads started.')
    build_pack(root, input_root)
    if auto_create_dataset:
        return save_private_dataset(root, owner, api)
    print('Save Version with output files, then attach through Add Input → Notebook Outputs.', flush=True)
    return None

# One startup cell. Processing starts only after Run in ComfyUI.
from pathlib import Path
import fcntl
import hashlib
import importlib.metadata as metadata
import json
import os
import re
import shutil
import signal
import socket
import subprocess
import sys
import time
import urllib.request

REQUIRE_ATTACHED_MODELS = True
ENABLE_SEEDVR2 = True
PORT = 8188
BASE = Path('/kaggle/temp/movie-studio-v1')
OUT = Path('/kaggle/working/movie-studio')
COMFY_REV = '1332fabfcc507aa7c548cd5152071ed494c207e8'
SEED_REV = '4490bd1f482e026674543386bb2a4d176da245b9'
CF_SHA = '1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872'
# Build embeds the catalog, custom IO code, and ordinary connected workflows here.
FILES = {'media.py': '"""Bounded SDR/CFR video IO and durable chunk checkpoints. No ML imports."""\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nfrom fractions import Fraction\n\nVERSION = \'movie-studio-v1\'\n_HASH_CACHE = {}\n\ndef run(args):\n    return subprocess.run(list(map(str, args)), check=True, capture_output=True).stdout\n\ndef digest(path):\n    stat = Path(path).stat()\n    key = (str(Path(path).resolve()), stat.st_dev, stat.st_ino, stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns)\n    if key in _HASH_CACHE:\n        return _HASH_CACHE[key]\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(8 * 1024**2), b\'\'):\n            h.update(block)\n    value = h.hexdigest()\n    _HASH_CACHE[key] = value\n    return value\n\ndef atomic_json(path, value):\n    path = Path(path)\n    temp = path.with_suffix(path.suffix + \'.partial\')\n    with open(temp, \'w\') as f:\n        json.dump(value, f, indent=2)\n        f.flush()\n        os.fsync(f.fileno())\n    temp.replace(path)\n\ndef probe(path, count=False):\n    args = [\'ffprobe\', \'-v\', \'error\']\n    if count:\n        args += [\'-count_frames\']\n    return json.loads(run(args + [\'-show_streams\', \'-show_format\', \'-of\', \'json\', path]))\n\ndef video_info(path):\n    data = probe(path)\n    videos = [s for s in data[\'streams\'] if s[\'codec_type\'] == \'video\' and not s.get(\'disposition\', {}).get(\'attached_pic\')]\n    if len(videos) != 1:\n        raise ValueError(\'Use a file with one video stream (excluding cover art).\')\n    v = videos[0]\n    if v.get(\'color_transfer\') in (\'smpte2084\', \'arib-std-b67\'):\n        raise ValueError(\'HDR needs a separate colour-managed workflow. This version accepts SDR.\')\n    if any(int(x.get(\'rotation\', 0)) % 360 for x in v.get(\'side_data_list\', [])):\n        raise ValueError(\'Rotate the source pixels before upscaling; display-rotation metadata is unsupported.\')\n    if v.get(\'sample_aspect_ratio\', \'1:1\') not in (\'1:1\', \'N/A\', \'0:1\'):\n        raise ValueError(\'Convert anamorphic video to square pixels before upscaling.\')\n    try:\n        fps = Fraction(v.get(\'avg_frame_rate\', \'0/1\'))\n    except (ValueError, ZeroDivisionError):\n        try:\n            fps = Fraction(v.get(\'r_frame_rate\', \'0/1\'))\n        except (ValueError, ZeroDivisionError):\n            raise ValueError(\'Unsupported or unknown source frame rate.\') from None\n    if not 0 < fps <= 120:\n        raise ValueError(\'Unsupported or unknown source frame rate.\')\n    # Scan timestamps incrementally, avoiding a whole-film JSON allocation.\n    cmd = [\'ffprobe\', \'-v\', \'error\', \'-select_streams\', str(v[\'index\']), \'-show_entries\',\n           \'frame=best_effort_timestamp_time\', \'-of\', \'csv=p=0\', str(path)]\n    n = 0\n    first = None\n    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True) as p:\n        for line in p.stdout:\n            value = line.strip().split(\',\')[0]\n            if not value:\n                continue\n            try:\n                timestamp = float(value)\n            except ValueError:\n                p.kill()\n                raise ValueError(\'Cannot determine frame timestamps.\')\n            if first is None:\n                first = timestamp\n            if abs(timestamp - first - n / float(fps)) > max(0.002, 0.1 / float(fps)):\n                p.kill()\n                raise ValueError(\'Variable-frame-rate video detected. Convert to constant frame rate first; no frames were silently retimed.\')\n            n += 1\n        if p.wait():\n            raise RuntimeError(\'ffprobe timestamp scan failed.\')\n    if n == 0:\n        raise ValueError(\'Video contains no readable frames.\')\n    return {\'width\': v[\'width\'], \'height\': v[\'height\'], \'fps\': str(fps), \'frames\': n,\n            \'start\': first, \'format_start\': float(data.get(\'format\', {}).get(\'start_time\', first)), \'stream_index\': v[\'index\'], \'audio_streams\': sum(s[\'codec_type\'] == \'audio\' for s in data[\'streams\']),\n            \'color\': {k: v.get(k) for k in (\'color_space\', \'color_transfer\', \'color_primaries\', \'color_range\')},\n            \'pixel_format\': v.get(\'pix_fmt\')}\n\ndef prepare(source, root, settings, core_frames, context, scale, max_edge):\n    source = Path(source).expanduser().resolve(strict=True)\n    if not source.is_file():\n        raise ValueError(\'Source must be a video file.\')\n    if not 1 <= core_frames <= 65 or not 0 <= context <= 8 or scale not in (2, 4):\n        raise ValueError(\'Invalid chunk, context or scale setting.\')\n    fingerprint = digest(source)\n    settings = dict(settings)\n    key = hashlib.sha256(json.dumps([VERSION, fingerprint, settings], sort_keys=True).encode()).hexdigest()[:24]\n    job = Path(root) / key\n    job.mkdir(parents=True, exist_ok=True)\n    manifest = job / \'job.json\'\n    if manifest.exists():\n        meta = json.loads(manifest.read_text())\n        if meta[\'source_sha256\'] != fingerprint or meta[\'settings\'] != settings:\n            raise RuntimeError(\'Checkpoint provenance mismatch.\')\n        meta[\'source\'] = str(source)  # same content can be attached at a different Kaggle path\n    else:\n        info = video_info(source)\n        w, h = info[\'width\'] * scale, info[\'height\'] * scale\n        if w % 2 or h % 2 or max(w, h) > max_edge:\n            raise ValueError(f\'Target {w}x{h} exceeds max_edge={max_edge}. Raise the explicit cap only after testing memory.\')\n        meta = {\'version\': VERSION, \'source\': str(source), \'source_sha256\': fingerprint,\n                \'settings\': settings, \'info\': info, \'scale\': scale, \'core_frames\': core_frames,\n                \'context\': context, \'chunks\': math.ceil(info[\'frames\'] / core_frames), \'job\': str(job)}\n        atomic_json(manifest, meta)\n    # Conservative uncompressed estimate. Never delete other jobs or user files.\n    frames = meta[\'info\'][\'frames\']\n    raw = frames * meta[\'info\'][\'width\'] * meta[\'info\'][\'height\'] * scale**2 * 3\n    existing = sum(p.stat().st_size for p in job.glob(\'chunk-*.mkv\'))\n    reserve = max(512 * 1024**2, raw // 10)\n    if shutil.disk_usage(job).free < max(0, raw - existing) + reserve:\n        raise RuntimeError(f\'Not enough disk for a conservative {raw / 1024**3:.2f} GiB checkpoint estimate plus reserve. Use a shorter clip or smaller target.\')\n    return meta\n\ndef bounds(meta, index):\n    start = index * meta[\'core_frames\']\n    end = min(start + meta[\'core_frames\'], meta[\'info\'][\'frames\'])\n    return start, end, max(0, start-meta[\'context\']), min(meta[\'info\'][\'frames\'], end+meta[\'context\'])\n\ndef committed(meta, index):\n    base = Path(meta[\'job\']) / f\'chunk-{index:06d}\'\n    receipt = base.with_suffix(\'.json\')\n    if not receipt.exists():\n        return False\n    record = json.loads(receipt.read_text())\n    movie = base.with_suffix(\'.mkv\')\n    if not movie.exists() or digest(movie) != record[\'sha256\']:\n        raise RuntimeError(f\'Corrupt committed checkpoint: {movie}. Preserve it and its receipt before repair.\')\n    return True\n\ndef next_chunk(meta):\n    for index in range(meta[\'chunks\']):\n        if not committed(meta, index):\n            return index\n    return None\n\ndef read_chunk(meta, index):\n    start, end, lo, hi = bounds(meta, index)\n    # Seek to a nearby keyframe, decode accurately, then select by original PTS.\n    # Half-frame boundaries tolerate container timebase rounding after CFR validation.\n    fps = float(Fraction(meta[\'info\'][\'fps\']))\n    first = meta[\'info\'][\'start\']\n    seek = max(0, first-meta[\'info\'].get(\'format_start\', first)+(lo-2)/fps) if lo > 2 else 0\n    lower, upper = first+(lo-0.5)/fps, first+(hi-0.5)/fps\n    args = [\'ffmpeg\', \'-v\', \'error\', \'-copyts\', \'-noautorotate\', \'-ss\', f\'{seek:.9f}\',\n            \'-i\', meta[\'source\'], \'-map\', f"0:{meta[\'info\'][\'stream_index\']}",\n            \'-vf\', f\'select=gte(t\\\\,{lower:.9f})*lt(t\\\\,{upper:.9f})\',\n            \'-fps_mode\', \'passthrough\', \'-frames:v\', str(hi-lo),\n            \'-f\', \'rawvideo\', \'-pix_fmt\', \'rgb24\', \'-threads\', \'1\', \'pipe:1\']\n    raw = run(args)\n    expected = (hi-lo)*meta[\'info\'][\'width\']*meta[\'info\'][\'height\']*3\n    if len(raw) != expected:\n        raise RuntimeError(f\'Expected {expected} decoded bytes, received {len(raw)}. No checkpoint saved.\')\n    return raw, hi-lo\n\ndef save_chunk(meta, index, raw, width, height, count):\n    start, end, lo, hi = bounds(meta, index)\n    if count != hi-lo or (width, height) != (meta[\'info\'][\'width\']*meta[\'scale\'], meta[\'info\'][\'height\']*meta[\'scale\']):\n        raise ValueError(\'Upscaler changed frame count or produced the wrong target dimensions.\')\n    stride = width*height*3\n    if len(raw) != count*stride:\n        raise ValueError(\'Incomplete output frame buffer.\')\n    # Context is reconstructed with neighbours, then discarded; no duplicate frames.\n    core = memoryview(raw)[(start-lo)*stride:(end-lo)*stride]\n    base = Path(meta[\'job\']) / f\'chunk-{index:06d}\'\n    temp = base.with_suffix(\'.partial.mkv\')\n    cmd = [\'ffmpeg\', \'-v\', \'error\', \'-y\', \'-f\', \'rawvideo\', \'-pix_fmt\', \'rgb24\', \'-s\', f\'{width}x{height}\',\n           \'-r\', meta[\'info\'][\'fps\'], \'-i\', \'pipe:0\', \'-an\', \'-c:v\', \'ffv1\', \'-level\', \'3\', \'-pix_fmt\', \'bgr0\', \'-threads\', \'1\', temp]\n    subprocess.run(list(map(str, cmd)), input=core, check=True, capture_output=True)\n    check = probe(temp, count=True)[\'streams\'][0]\n    if int(check[\'nb_read_frames\']) != end-start:\n        raise RuntimeError(\'Checkpoint verification failed.\')\n    temp.replace(base.with_suffix(\'.mkv\'))\n    atomic_json(base.with_suffix(\'.json\'), {\'sha256\': digest(base.with_suffix(\'.mkv\')), \'start\': start, \'end\': end,\n                                           \'width\': width, \'height\': height})\n\ndef export(meta, container=\'mp4\'):\n    if container not in (\'mp4\', \'mkv\'):\n        raise ValueError(\'Choose mp4 or mkv.\')\n    if next_chunk(meta) is not None:\n        raise RuntimeError(\'Cannot export incomplete chunks.\')\n    job = Path(meta[\'job\'])\n    target = job / (\'upscaled.\'+container)\n    receipt = job / (\'export-\'+container+\'.json\')\n    if receipt.exists():\n        old = json.loads(receipt.read_text())\n        if target.exists() and digest(target) == old[\'sha256\']:\n            return target\n        raise RuntimeError(\'Completed export is missing or corrupt; preserve its receipt before repair.\')\n    listing = job / \'concat.txt\'\n    listing.write_text(\'\'.join(f"file \'chunk-{i:06d}.mkv\'\\n" for i in range(meta[\'chunks\'])))\n    temp = job / (\'upscaled.partial.\'+container)\n    args = [\'ffmpeg\', \'-v\', \'error\', \'-y\', \'-copyts\', \'-f\', \'concat\', \'-safe\', \'1\', \'-i\', listing,\n            \'-itsoffset\', str(-meta[\'info\'][\'start\']), \'-i\', meta[\'source\'], \'-map\', \'0:v:0\', \'-map\', \'1:a?\',\n            \'-map_metadata\', \'1\', \'-map_chapters\', \'1\', \'-c:v\', \'libx264\', \'-preset\', \'medium\', \'-crf\', \'18\',\n            \'-pix_fmt\', \'yuv420p\', \'-r\', meta[\'info\'][\'fps\'], \'-fps_mode\', \'cfr\', \'-c:a\', \'copy\']\n    color = meta[\'info\'].get(\'color\', {})\n    matrix = color.get(\'color_space\')\n    if matrix in (\'bt709\', \'bt470bg\', \'smpte170m\', \'smpte240m\', \'bt2020nc\'):\n        args += [\'-vf\', \'scale=out_color_matrix=\'+matrix+\':out_range=tv\', \'-colorspace\', matrix, \'-color_range\', \'tv\']\n    for key, flag in [(\'color_transfer\', \'-color_trc\'), (\'color_primaries\', \'-color_primaries\')]:\n        value = color.get(key)\n        if value and value not in (\'unknown\', \'reserved\', \'unspecified\'):\n            args += [flag, value]\n    if container == \'mkv\':\n        args += [\'-map\', \'1:s?\', \'-c:s\', \'copy\']\n    else:\n        args += [\'-movflags\', \'+faststart\']\n    audio_mode = \'copy\'\n    try:\n        run(args + [temp])\n    except subprocess.CalledProcessError:\n        if container != \'mp4\':\n            raise\n        audio_mode = \'aac 192k (MP4 stream-copy failed)\'\n        retry = list(args)\n        at = retry.index(\'-c:a\')\n        retry[at+1] = \'aac\'\n        run(retry + [\'-b:a\', \'192k\', temp])\n    info = probe(temp, count=True)\n    v = next(s for s in info[\'streams\'] if s[\'codec_type\'] == \'video\')\n    actual_audio = sum(s[\'codec_type\'] == \'audio\' for s in info[\'streams\'])\n    if int(v[\'nb_read_frames\']) != meta[\'info\'][\'frames\'] or actual_audio != meta[\'info\'][\'audio_streams\']:\n        raise RuntimeError(\'Final frame/audio-stream count failed validation; partial file retained.\')\n    if (v[\'width\'], v[\'height\']) != (meta[\'info\'][\'width\']*meta[\'scale\'], meta[\'info\'][\'height\']*meta[\'scale\']):\n        raise RuntimeError(\'Final video dimensions failed validation.\')\n    temp.replace(target)\n    atomic_json(receipt, {\'file\': str(target), \'sha256\': digest(target), \'audio_mode\': audio_mode,\n                          \'frames\': int(v[\'nb_read_frames\']), \'fps\': meta[\'info\'][\'fps\']})\n    return target\n', 'nodes.py': '"""Only video IO/checkpoint nodes; the canvas uses real upstream upscale nodes."""\nimport copy\nimport json\nimport os\nfrom pathlib import Path\nimport urllib.request\nimport numpy as np\nimport torch\nimport folder_paths\nfrom . import media\n\nclass MovieChunk:\n    CATEGORY = \'Movie Studio\'\n    RETURN_TYPES = (\'IMAGE\', \'MOVIE_JOB\', \'INT\')\n    RETURN_NAMES = (\'frames\', \'job\', \'target_short_edge\')\n    FUNCTION = \'load\'\n\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\'required\': {\n            \'source_path\': (\'STRING\', {\'default\': \'/kaggle/input/your-video/clip.mp4\'}),\n            \'scale\': ([2, 4], {\'default\': 2}),\n            \'chunk_frames\': (\'INT\', {\'default\': 17, \'min\': 1, \'max\': 65}),\n            \'context_frames\': (\'INT\', {\'default\': 4, \'min\': 0, \'max\': 8}),\n            \'max_target_edge\': (\'INT\', {\'default\': 1920, \'min\': 64, \'max\': 7680}),\n            \'resume_tick\': (\'INT\', {\'default\': 0, \'min\': 0, \'max\': 2147483647}),\n        }, \'hidden\': {\'prompt\': \'PROMPT\', \'unique_id\': \'UNIQUE_ID\'}}\n\n    @classmethod\n    def IS_CHANGED(cls, **kwargs):\n        return float(\'nan\')\n\n    def load(self, source_path, scale, chunk_frames, context_frames, max_target_edge, resume_tick, prompt, unique_id):\n        # Exclude queue counters and path identity; file SHA is part of the job key.\n        settings = copy.deepcopy(prompt)\n        for node in settings.values():\n            node.pop(\'_meta\', None)\n            if node[\'class_type\'] == \'MovieStudioChunk\':\n                node[\'inputs\'].pop(\'resume_tick\', None)\n                node[\'inputs\'].pop(\'source_path\', None)\n            if node[\'class_type\'] == \'MovieStudioCommit\':\n                node[\'inputs\'].pop(\'auto_continue\', None)\n                node[\'inputs\'].pop(\'container\', None)\n        root = Path(folder_paths.get_output_directory()) / \'movie-jobs\'\n        meta = media.prepare(source_path, root, settings, chunk_frames, context_frames, int(scale), max_target_edge)\n        index = media.next_chunk(meta)\n        if index is None:\n            # Re-export if the cell stopped after committing the last chunk.\n            output = next(n for n in prompt.values() if n[\'class_type\'] == \'MovieStudioCommit\')\n            target = media.export(meta, output[\'inputs\'][\'container\'])\n            raise RuntimeError(f\'Job already complete: {target}. Change settings or source to start a different job.\')\n        raw, count = media.read_chunk(meta, index)\n        frames = np.frombuffer(raw, dtype=np.uint8).reshape(count, meta[\'info\'][\'height\'], meta[\'info\'][\'width\'], 3).copy()\n        meta = dict(meta, index=index, source_node=str(unique_id), resume_tick=resume_tick)\n        print(f"Movie Studio: chunk {index+1}/{meta[\'chunks\']}, job {meta[\'job\']}", flush=True)\n        return torch.from_numpy(frames).float().div_(255), meta, min(meta[\'info\'][\'width\'], meta[\'info\'][\'height\'])*int(scale)\n\nclass MovieCommit:\n    CATEGORY = \'Movie Studio\'\n    OUTPUT_NODE = True\n    RETURN_TYPES = (\'STRING\',)\n    RETURN_NAMES = (\'status\',)\n    FUNCTION = \'save\'\n\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\'required\': {\'frames\': (\'IMAGE\',), \'job\': (\'MOVIE_JOB\',),\n                             \'container\': ([\'mp4\', \'mkv\'], {\'default\': \'mp4\'}),\n                             \'auto_continue\': (\'BOOLEAN\', {\'default\': True})},\n                \'hidden\': {\'prompt\': \'PROMPT\', \'extra_pnginfo\': \'EXTRA_PNGINFO\'}}\n\n    def save(self, frames, job, container, auto_continue, prompt, extra_pnginfo=None):\n        import comfy.model_management as mm\n        mm.throw_exception_if_processing_interrupted()\n        # Keep float conversion bounded to one frame instead of doubling the full chunk.\n        count, h, w, channels = frames.shape\n        if channels != 3:\n            raise ValueError(\'Expected RGB frames.\')\n        raw = bytearray(count*h*w*3)\n        stride = h*w*3\n        for i in range(count):\n            raw[i*stride:(i+1)*stride] = (frames[i].detach().cpu().clamp(0, 1).numpy()*255).round().astype(np.uint8).tobytes()\n        media.save_chunk(job, job[\'index\'], raw, w, h, count)\n        pending = media.next_chunk(job)\n        if pending is None:\n            mm.throw_exception_if_processing_interrupted()\n            target = media.export(job, container)\n            message = f\'COMPLETE: {target}\'\n        elif auto_continue:\n            mm.throw_exception_if_processing_interrupted()\n            # One ordinary ComfyUI job per chunk. Clearing the queue stops continuation;\n            # interrupting the current chunk stops before committing/queuing another.\n            again = copy.deepcopy(prompt)\n            again[job[\'source_node\']][\'inputs\'][\'resume_tick\'] = (job[\'resume_tick\']+1) % 2147483647\n            payload = {\'prompt\': again, \'extra_data\': {\'extra_pnginfo\': extra_pnginfo or {}}}\n            req = urllib.request.Request(os.environ.get(\'MOVIE_STUDIO_URL\', \'http://127.0.0.1:8188\')+\'/prompt\',\n                                         data=json.dumps(payload).encode(), headers={\'Content-Type\': \'application/json\'})\n            with urllib.request.urlopen(req, timeout=30) as response:\n                result = json.load(response)\n            if \'prompt_id\' not in result or result.get(\'node_errors\'):\n                raise RuntimeError(f\'Chunk saved, but continuation failed: {result}\')\n            message = f"Saved chunk {job[\'index\']+1}/{job[\'chunks\']}; queued next. {job[\'job\']}"\n        else:\n            message = f"PAUSED after chunk {job[\'index\']+1}/{job[\'chunks\']}. Press Run to resume. {job[\'job\']}"\n        print(message, flush=True)\n        return {\'ui\': {\'text\': [message]}, \'result\': (message,)}\n\nNODE_CLASS_MAPPINGS = {\'MovieStudioChunk\': MovieChunk, \'MovieStudioCommit\': MovieCommit}\nNODE_DISPLAY_NAME_MAPPINGS = {\'MovieStudioChunk\': \'Movie • Load next resumable chunk\',\n                              \'MovieStudioCommit\': \'Movie • Commit chunk / export with audio\'}\n', '__init__.py': 'from .nodes import NODE_CLASS_MAPPINGS, NODE_DISPLAY_NAME_MAPPINGS\nWEB_DIRECTORY = "web"\n', 'web/open.js': 'import { app } from "../../scripts/app.js";\napp.registerExtension({\n  name: "MovieStudio.Open",\n  setup() {\n    const url = new URL(window.location.href);\n    if (url.searchParams.get("movie_studio") !== "v1") return;\n    void (async () => {\n      try {\n        const deadline = Date.now() + 60000;\n        const canvasReady = () => { try { return !!app.canvas?.canvas; } catch { return false; } };\n        while (!app.vueAppReady || !app.positionConversion || !canvasReady() || !window.LiteGraph?.registered_node_types?.MovieStudioChunk || !window.LiteGraph?.registered_node_types?.MovieStudioCommit) {\n          if (Date.now() > deadline) throw new Error("ComfyUI did not finish starting");\n          await new Promise(r => setTimeout(r, 100));\n        }\n        await new Promise(r => setTimeout(r, 500));\n        const response = await fetch(new URL("default.json", import.meta.url), {cache: "no-store"});\n        if (!response.ok) throw new Error("Workflow could not be loaded");\n        await app.loadGraphData(await response.json(), true, true, "Movie Studio.json");\n        url.searchParams.delete("movie_studio");\n        history.replaceState(null, "", url);\n      } catch (error) { window.alert(`Movie Studio: ${error.message}`); }\n    })();\n  }\n});\n'}
WORKFLOWS = {'Movie Studio - SeedVR2': {'last_node_id': 5, 'last_link_id': 6, 'nodes': [{'id': 1, 'type': 'MovieStudioChunk', 'pos': [0, 0], 'size': [340, 390], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'frames', 'type': 'IMAGE', 'links': [1]}, {'name': 'job', 'type': 'MOVIE_JOB', 'links': [6]}, {'name': 'target_short_edge', 'type': 'INT', 'links': [4]}], 'properties': {'Node name for S&R': 'MovieStudioChunk'}, 'widgets_values': ['/kaggle/input/your-video/clip.mp4', 2, 17, 4, 1920, 0], 'title': '1 • Set movie path; 2× default; resumes saved chunks'}, {'id': 2, 'type': 'SeedVR2LoadDiTModel', 'pos': [390, 0], 'size': [340, 390], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SEEDVR2_DIT', 'type': 'SEEDVR2_DIT', 'links': [2]}], 'properties': {'Node name for S&R': 'SeedVR2LoadDiTModel'}, 'widgets_values': ['seedvr2_ema_3b-Q8_0.gguf', 'cuda:0', 32, True, 'cpu', True, 'sdpa'], 'title': '2 • 3B Q8 / CPU block swapping / T4'}, {'id': 3, 'type': 'SeedVR2LoadVAEModel', 'pos': [390, 450], 'size': [340, 390], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SEEDVR2_VAE', 'type': 'SEEDVR2_VAE', 'links': [3]}], 'properties': {'Node name for S&R': 'SeedVR2LoadVAEModel'}, 'widgets_values': ['ema_vae_fp16.safetensors', 'cuda:0', True, 512, 64, True, 512, 64, 'false', 'cpu', True], 'title': '3 • FP16 VAE; tiled encode and decode'}, {'id': 4, 'type': 'SeedVR2VideoUpscaler', 'pos': [800, 0], 'size': [340, 390], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 1}, {'name': 'dit', 'type': 'SEEDVR2_DIT', 'link': 2}, {'name': 'vae', 'type': 'SEEDVR2_VAE', 'link': 3}, {'name': 'resolution', 'type': 'INT', 'link': 4, 'widget': {'name': 'resolution'}}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [5]}], 'properties': {'Node name for S&R': 'SeedVR2VideoUpscaler'}, 'widgets_values': [42, 'fixed', 1024, 0, 5, True, 'lab', 2, 0, 0.0, 0.0, 'cpu', True], 'title': '4 • Temporal video restoration; target comes from source × scale'}, {'id': 5, 'type': 'MovieStudioCommit', 'pos': [1210, 0], 'size': [340, 390], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'frames', 'type': 'IMAGE', 'link': 5}, {'name': 'job', 'type': 'MOVIE_JOB', 'link': 6}], 'outputs': [{'name': 'status', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'MovieStudioCommit'}, 'widgets_values': ['mp4', True], 'title': '5 • Save chunk → continue → final movie with original audio'}], 'links': [[1, 1, 0, 4, 0, 'IMAGE'], [2, 2, 0, 4, 1, 'SEEDVR2_DIT'], [3, 3, 0, 4, 2, 'SEEDVR2_VAE'], [4, 1, 2, 4, 3, 'INT'], [5, 4, 0, 5, 0, 'IMAGE'], [6, 1, 1, 5, 1, 'MOVIE_JOB']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [80, 80]}}, 'version': 0.4}, 'Movie Studio - RealESRGAN': {'last_node_id': 5, 'last_link_id': 5, 'nodes': [{'id': 1, 'type': 'MovieStudioChunk', 'pos': [0, 0], 'size': [340, 390], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'frames', 'type': 'IMAGE', 'links': [2]}, {'name': 'job', 'type': 'MOVIE_JOB', 'links': [5]}, {'name': 'target_short_edge', 'type': 'INT', 'links': []}], 'properties': {'Node name for S&R': 'MovieStudioChunk'}, 'widgets_values': ['/kaggle/input/your-video/clip.mp4', 2, 17, 0, 1920, 0], 'title': '1 • Set movie path; 2× default; resumes saved chunks'}, {'id': 2, 'type': 'UpscaleModelLoader', 'pos': [390, 420], 'size': [340, 390], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'UPSCALE_MODEL', 'type': 'UPSCALE_MODEL', 'links': [1]}], 'properties': {'Node name for S&R': 'UpscaleModelLoader'}, 'widgets_values': ['RealESRGAN_x4plus.pth'], 'title': '2 • Verified Real-ESRGAN checkpoint'}, {'id': 3, 'type': 'ImageUpscaleWithModel', 'pos': [390, 0], 'size': [340, 390], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'upscale_model', 'type': 'UPSCALE_MODEL', 'link': 1}, {'name': 'image', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'ImageUpscaleWithModel'}, 'widgets_values': [], 'title': '3 • Native tiled 4× model'}, {'id': 4, 'type': 'ImageScaleBy', 'pos': [800, 0], 'size': [340, 390], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}], 'properties': {'Node name for S&R': 'ImageScaleBy'}, 'widgets_values': ['lanczos', 0.5], 'title': '4 • 4× → 2×; set scale_by=1 for a 4× job'}, {'id': 5, 'type': 'MovieStudioCommit', 'pos': [1210, 0], 'size': [340, 390], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'frames', 'type': 'IMAGE', 'link': 4}, {'name': 'job', 'type': 'MOVIE_JOB', 'link': 5}], 'outputs': [{'name': 'status', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'MovieStudioCommit'}, 'widgets_values': ['mp4', True], 'title': '5 • Save chunk → continue → final movie with original audio'}], 'links': [[1, 2, 0, 3, 0, 'UPSCALE_MODEL'], [2, 1, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE'], [4, 4, 0, 5, 0, 'IMAGE'], [5, 1, 1, 5, 1, 'MOVIE_JOB']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [80, 80]}}, 'version': 0.4}}
CF_SIZE = 40274445


BASE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
lock = open(BASE/'startup.lock', 'w')
try:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
except BlockingIOError:
    raise RuntimeError('Movie Studio is already running; stop its earlier startup cell.')
children = []

def run(args, **kwargs):
    subprocess.run(list(map(str, args)), check=True, **kwargs)

def checkout(url, dest, revision):
    if not dest.exists():
        run(['git', 'clone', '--no-checkout', url, dest])
        run(['git', '-C', dest, 'checkout', '--detach', revision])
    actual = subprocess.check_output(['git', '-C', str(dest), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != revision:
        raise RuntimeError(f'Unexpected source revision at {dest}; preserved.')

def spawn(args, log, env=None):
    with open(log, 'w') as f:
        p = subprocess.Popen(list(map(str, args)), stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)
    children.append(p)
    return p

try:
    with socket.socket() as check:
        try:
            check.bind(('127.0.0.1', PORT))
        except OSError:
            raise RuntimeError('Port occupied. Stop the earlier Image/Movie/H3 studio cell first.')
    required = [a for a in ASSETS if ENABLE_SEEDVR2 or a['folder'] != 'SEEDVR2']
    found = {}
    for asset in required:
        dest = BASE/'ComfyUI/models'/asset['folder']/asset['name']
        if dest.exists() and not valid(dest, asset):
            raise RuntimeError(f'Model checksum mismatch; preserved: {dest}')
        candidate = dest.resolve() if dest.exists() else find_asset(asset, ['/kaggle/input'])
        if candidate:
            found[asset['name']] = candidate
        elif REQUIRE_ATTACHED_MODELS:
            raise RuntimeError('Missing verified model '+asset['name']+'. Attach the automatically created Movie Upscaler Dataset via Add Input → Your Datasets, or attach saved Notebook Outputs. No model downloads started.')
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        if os.geteuid() == 0 and shutil.which('apt-get'):
            run(['apt-get', '-qq', 'update'])
            run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('FFmpeg/ffprobe installation was unavailable in this Kaggle image.')
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Select Kaggle GPU T4 and rerun. Model-pack preparation uses Accelerator: None.')
    free, total = torch.cuda.mem_get_info(0)
    if free < 10*1024**3:
        raise RuntimeError('GPU 0 needs at least 10 GiB free; stop other GPU processes first.')
    print('GPU 0:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__, flush=True)
    print('This version uses GPU 0. A second T4 does not pool its memory.', flush=True)
    comfy = BASE/'ComfyUI'
    checkout('https://github.com/comfyanonymous/ComfyUI.git', comfy, COMFY_REV)
    seed = comfy/'custom_nodes/SeedVR2'
    if ENABLE_SEEDVR2:
        checkout('https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git', seed, SEED_REV)
    python = BASE/'venv/bin/python'
    if not python.exists():
        run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', python.parent.parent])
    protected = ['torch', 'torchvision', 'torchaudio', 'numpy', 'scipy']
    constraints = BASE/'constraints.txt'
    constraints.write_text('\n'.join(f'{p}=={metadata.version(p)}' for p in protected))
    requirements = BASE/'requirements.txt'
    text = (comfy/'requirements.txt').read_text()
    if ENABLE_SEEDVR2:
        text += '\n'+(seed/'requirements.txt').read_text()
    requirements.write_text(text)
    fingerprint = hashlib.sha256((text+constraints.read_text()+sys.version+SEED_REV).encode()).hexdigest()
    marker = BASE/'setup.complete'
    if not marker.exists() or marker.read_text() != fingerprint:
        run([sys.executable, '-m', 'pip', '--python', python, 'install', '-r', requirements, '-c', constraints])
        run([python, '-c', 'import torch, torchvision, numpy, cv2, aiohttp; assert torch.cuda.is_available()'])
        marker.write_text(fingerprint)
    for asset in required:
        dest = comfy/'models'/asset['folder']/asset['name']
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            if asset['name'] in found:
                dest.symlink_to(found[asset['name']])
            else:
                fetch(asset, dest)
    custom = comfy/'custom_nodes/MovieStudio'
    custom.mkdir(parents=True, exist_ok=True)
    for name, text in FILES.items():
        target = custom/name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(text)
    workflows = {k: v for k, v in WORKFLOWS.items() if ENABLE_SEEDVR2 or 'SeedVR2' not in k}
    wf_dir = comfy/'user/default/workflows'
    wf_dir.mkdir(parents=True, exist_ok=True)
    for name, graph in workflows.items():
        (wf_dir/(name+'.json')).write_text(json.dumps(graph))
    default = 'Movie Studio - SeedVR2' if ENABLE_SEEDVR2 else 'Movie Studio - RealESRGAN'
    (custom/'web/default.json').write_text(json.dumps(workflows[default]))
    cf = BASE/'cloudflared'
    if not cf.exists():
        fetch({'url': 'https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64',
               'bytes': CF_SIZE, 'sha256': CF_SHA}, cf)
    if sha(cf) != CF_SHA:
        raise RuntimeError('cloudflared checksum mismatch.')
    cf.chmod(0o700)
    env = os.environ.copy()
    env['MOVIE_STUDIO_URL'] = f'http://127.0.0.1:{PORT}'
    server = spawn([python, comfy/'main.py', '--listen', '127.0.0.1', '--port', PORT, '--lowvram', '--force-fp16',
                    '--preview-method', 'none', '--cache-none', '--output-directory', OUT, '--disable-auto-launch'], OUT/'comfy.log', env)
    deadline = time.monotonic()+240
    while True:
        if server.poll() is not None:
            raise RuntimeError('ComfyUI stopped. See /kaggle/working/movie-studio/comfy.log.')
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/object_info', timeout=3) as response:
                info = json.load(response)
            break
        except OSError:
            if time.monotonic() > deadline:
                raise RuntimeError('ComfyUI startup timed out; see comfy.log.')
            time.sleep(1)
    for graph in workflows.values():
        for node in graph['nodes']:
            if node['type'] not in info:
                raise RuntimeError('Required node unavailable: '+node['type']+'. See comfy.log.')
    (OUT/'provenance.json').write_text(json.dumps({'comfy': COMFY_REV, 'seedvr2': SEED_REV, 'assets': required,
                                                  'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)}, indent=2))
    with open(OUT/'packages.txt', 'w') as f:
        run([sys.executable, '-m', 'pip', '--python', python, 'freeze'], stdout=f)
    tunnel = spawn([cf, 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'], OUT/'tunnel.log')
    deadline = time.monotonic()+90
    url = None
    while time.monotonic() < deadline:
        matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', (OUT/'tunnel.log').read_text())
        if matches:
            url = matches[-1]+'/?movie_studio=v1'
            break
        if tunnel.poll() is not None:
            raise RuntimeError('Cloudflare stopped; see tunnel.log.')
        time.sleep(1)
    if not url:
        raise RuntimeError('No studio URL received; see tunnel.log.')
    (OUT/'studio-url.txt').write_text(url)
    print('\nOPEN MOVIE STUDIO:', url, flush=True)
    print('Set source_path in the leftmost node to your attached MP4/MKV, then press Run.')
    videos = [p for p in Path('/kaggle/input').rglob('*') if p.is_file() and p.suffix.lower() in ('.mp4', '.mkv', '.mov', '.webm', '.avi')]
    for movie in videos[:20]: print('Attached video:', movie)
    print('No processing has started. Use queue count 1; turn OFF frontend Auto Queue.')
    print('To pause: Interrupt, then Clear Queue. Press Run with the same settings to resume.')
    print('Saved chunks and final movies:', OUT/'movie-jobs')
    print('Keep this cell running. Save Version with output files before ending the session.')
    print('Anyone with this temporary session URL can access the studio.', flush=True)
    while server.poll() is None and tunnel.poll() is None:
        time.sleep(2)
    raise RuntimeError('Studio or tunnel stopped; inspect saved logs.')
finally:
    for p in reversed(children):
        if p.poll() is None:
            os.killpg(p.pid, signal.SIGTERM)
            try:
                p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(p.pid, signal.SIGKILL)
                p.wait()
    fcntl.flock(lock, fcntl.LOCK_UN)
    lock.close()
